In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_epochs = 80
learnig_rate = 0.001

transform = transforms.Compose([
    transforms.Pad(4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32),
    transforms.ToTensor()])

In [ ]:
train_dataset = torchvision.datasets.CIFAR10(root = '../../data', train = True, download = True, transform = transform)
test_dataset = torchvision.datasets.CIFAR10(root = '../../data', train = False, download = True, transform = transforms.ToTensor())



train_loader = torch.utils.data.DataLoader(dataset = train_dataset, batch_size = 100, shuffle = True)
test_loader = torch.utils.data.DataLoader(dataset = test_dataset, batch_size = 100, shuffle = False)

100%|██████████| 170M/170M [00:03<00:00, 44.7MB/s]


In [ ]:
class VGG(nn.Module):
  def __init__(self):
    super(VGG, self).__init__()
    self.maxpool = nn.MaxPool2d(4)
    self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1)
    self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1)
    self.relu = nn.ReLU(inplace=True)
    self.fc = nn.Linear(1568, 10)

  def forward(self, x):
    x = self.relu(self.conv1(x))
    x = self.relu(self.conv2(x))
    x = self.maxpool(x)
    x = x.view(x.size(0), -1)
    x = self.fc(x)

    return x



model = VGG().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learnig_rate)

def update_lr(optimizer, lr):
  for param_group in optimizer.param_groups:
    param_group['lr'] = lr

1. Training/Test를 for문 안에서 함께 수행하도록 코드를 변경하시오.

In [ ]:
total_step = len(train_loader)
curr_lr = learnig_rate

def train(epoch):
    model.train()

    for i, (images, labels) in enumerate(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 100 == 0:
            print('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'.format(epoch,num_epochs,i+1,total_step,loss.item()))

    if (epoch+1) % 20 == 0:
        curr_lr /= 3
        update_lr(optimizer, curr_lr)

In [ ]:
def test():
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print('Test Accuracy: {} %'.format(100 * correct / total))

In [ ]:
for epoch in range(1, 10):
  train(epoch)
  test()

Epoch [1/80], Step [100/500], Loss: 1.9122
Epoch [1/80], Step [200/500], Loss: 1.8665
Epoch [1/80], Step [300/500], Loss: 1.7067
Epoch [1/80], Step [400/500], Loss: 1.7492
Epoch [1/80], Step [500/500], Loss: 1.7212
Test Accuracy: 49.26 %
Epoch [2/80], Step [100/500], Loss: 1.5566
Epoch [2/80], Step [200/500], Loss: 1.3643
Epoch [2/80], Step [300/500], Loss: 1.3348
Epoch [2/80], Step [400/500], Loss: 1.4766
Epoch [2/80], Step [500/500], Loss: 1.4029
Test Accuracy: 53.08 %
Epoch [3/80], Step [100/500], Loss: 1.3538
Epoch [3/80], Step [200/500], Loss: 1.4122
Epoch [3/80], Step [300/500], Loss: 1.3830
Epoch [3/80], Step [400/500], Loss: 1.2468
Epoch [3/80], Step [500/500], Loss: 1.1960
Test Accuracy: 55.94 %
Epoch [4/80], Step [100/500], Loss: 1.3383
Epoch [4/80], Step [200/500], Loss: 1.3212
Epoch [4/80], Step [300/500], Loss: 1.2730
Epoch [4/80], Step [400/500], Loss: 1.2146
Epoch [4/80], Step [500/500], Loss: 1.1824
Test Accuracy: 58.11 %
Epoch [5/80], Step [100/500], Loss: 1.4809
Epoch

2. 데이터 전처리로 정규화(normalization)를 수행하시오.

In [ ]:
norm_transform = transforms.Compose([transforms.ToTensor()])

norm_dataset = torchvision.datasets.CIFAR10(
    root='../../data',
    train=True,
    download=True,
    transform=norm_transform
)

norm_loader = torch.utils.data.DataLoader(
    norm_dataset,
    batch_size=100,
    shuffle=False
)

mean = 0.
std = 0.
total_images = 0

for images, _ in norm_loader:

    batch_samples = images.size(0)
    images = images.view(batch_samples,images.size(1),-1)

    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    total_images += batch_samples

mean /= total_images
std /= total_images

In [ ]:
transform = transforms.Compose([
    transforms.Pad(4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(),
                         std.tolist())
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(),
                         std.tolist())
])

3. 아래 VGG모델을 구현하고 CIFAR-100 데이터를 학습하시오. 학습 결과
를 report 하시오

In [ ]:
train_dataset = torchvision.datasets.CIFAR100(root = '../../data', train = True, download = True, transform = transform)
test_dataset = torchvision.datasets.CIFAR100(root = '../../data', train = False, download = True, transform = transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(dataset = train_dataset, batch_size = 100, shuffle = True)
test_loader = torch.utils.data.DataLoader(dataset = test_dataset, batch_size = 100, shuffle = False)

100%|██████████| 169M/169M [00:03<00:00, 47.2MB/s]


In [ ]:
class VGG3(nn.Module):

    def __init__(self):
        super(VGG3, self).__init__()

        self.relu = nn.ReLU()


        self.block2_1 = nn.Conv2d(3, 64,kernel_size=3,padding=1)
        self.block2_2 = nn.Conv2d(64, 128,kernel_size=3,padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.block3_1 = nn.Conv2d(128, 512,kernel_size=3,padding=1)
        self.block3_2 = nn.Conv2d(512, 512,kernel_size=3,padding=1)
        self.block3_3 = nn.Conv2d(512, 512,kernel_size=3,padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2,stride=2)

        self.fc1 = nn.Linear(512 * 8 * 8, 512)
        self.fc2 = nn.Linear(512,100)

    def forward(self, x):
        x = self.relu(self.block2_1(x))
        x = self.relu(self.block2_2(x))
        x = self.pool1(x)

        x = self.relu(self.block3_1(x))
        x = self.relu(self.block3_2(x))
        x = self.relu(self.block3_3(x))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x

4. 하이퍼파라미터 튜닝

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def cutmix(data, targets, alpha=1.0):

    indices = torch.randperm(data.size(0))
    shuffled_data = data[indices]
    shuffled_targets = targets[indices]

    lam = np.random.beta(alpha, alpha)

    B, C, H, W = data.size()

    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    data[:, :, y1:y2, x1:x2] = \
        shuffled_data[:, :, y1:y2, x1:x2]

    lam = 1 - ((x2-x1)*(y2-y1)/(W*H))

    return data, targets, shuffled_targets, lam

class VGG3(nn.Module):

    def __init__(self, ch1=64, ch2=128, ch3=256, stride=2):
        super(VGG3, self).__init__()

        self.relu = nn.ReLU()
        self.block2_1 = nn.Conv2d(3, ch1, 3, padding=1)
        self.block2_2 = nn.Conv2d(ch1, ch2, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2, stride=stride)
        self.block3_1 = nn.Conv2d(ch2, ch3, 3, padding=1)
        self.block3_2 = nn.Conv2d(ch3, ch3, 3, padding=1)
        self.block3_3 = nn.Conv2d(ch3, ch3, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, stride=stride)
        self.fc1 = nn.Linear(ch3 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 100)

    def forward(self, x):

        x = self.relu(self.block2_1(x))
        x = self.relu(self.block2_2(x))
        x = self.pool1(x)

        x = self.relu(self.block3_1(x))
        x = self.relu(self.block3_2(x))
        x = self.relu(self.block3_3(x))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

experiments = [
    {"ch1":64, "ch2":128, "ch3":256,
     "stride":2, "crop":32, "cutmix":False},
    {"ch1":32, "ch2":64, "ch3":128,
     "stride":2, "crop":32, "cutmix":False},
    {"ch1":64, "ch2":128, "ch3":256,
     "stride":1, "crop":32, "cutmix":False},
    {"ch1":64, "ch2":128, "ch3":256,
     "stride":2, "crop":24, "cutmix":False},
    {"ch1":64, "ch2":128, "ch3":256,
     "stride":2, "crop":32, "cutmix":True}
]

num_epochs = 10

results = []

for i, exp in enumerate(experiments):

    print(f"\n===== Experiment {i+1} =====")

    transform_train = transforms.Compose([
        transforms.RandomCrop(exp["crop"]),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
    ])

    trainset = torchvision.datasets.CIFAR100(
        root='./data',
        train=True,
        download=True,
        transform=transform_train)

    testset = torchvision.datasets.CIFAR100(
        root='./data',
        train=False,
        download=True,
        transform=transform_test)

    train_loader = torch.utils.data.DataLoader(
        trainset,
        batch_size=128,
        shuffle=True)

    test_loader = torch.utils.data.DataLoader(
        testset,
        batch_size=128,
        shuffle=False)

    model = VGG3(
        exp["ch1"],
        exp["ch2"],
        exp["ch3"],
        exp["stride"]
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    params = count_parameters(model)
    start_time = time.time()

    for epoch in range(num_epochs):
        model.train()
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            if exp["cutmix"]:
                images, labels_a, labels_b, lam = \
                    cutmix(images, labels)
                outputs = model(images)
                loss = lam * criterion(outputs, labels_a) + \
                       (1 - lam) * criterion(outputs, labels_b)

            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


    end_time = time.time()
    training_time = end_time - start_time

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    results.append([
        i+1,
        exp["ch1"],
        exp["stride"],
        exp["crop"],
        exp["cutmix"],
        params,
        training_time,
        accuracy
    ])

print("\n===== Final Results =====")
print("Exp | Ch | Stride | Crop | CutMix | Params | Time | Acc")

for r in results:
    print(r)